# Week 3 — ML Model Training & Evaluation
**AI-Powered WAF | Random Forest Classifier**

Steps:
1. Load `data/processed.csv`
2. Train/test split + scaling
3. Train Random Forest (200 trees)
4. 5-Fold cross-validation
5. Evaluate: accuracy, precision, recall, F1, ROC-AUC
6. Confusion matrix, ROC curve, feature importance
7. Threshold analysis
8. SHAP explainability
9. Save model and scaler

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import numpy as np

print('Libraries loaded OK')

Libraries loaded OK


## 1. Load Processed Data

In [2]:
df = pd.read_csv('../data/processed.csv')
X  = df.drop(columns=['label'])
y  = df['label']

print(f'Shape: {df.shape}')
print(f'Normal : {(y==0).sum():,}  |  Attack: {(y==1).sum():,}')
X.head()

Shape: (61065, 16)
Normal : 36,000  |  Attack: 25,065


,method_is_post,url_length,path_depth,query_length,num_query_params,body_length,num_body_params,content_length,has_cookie,has_sql,has_xss,has_path_traversal,has_cmd_injection,has_null_byte,special_char_count
0,0,56,3,0,0,0,0,0,1,0,0,0,0,0,0
1,0,281,3,230,13,0,0,0,1,0,0,0,0,0,13
2,0,58,4,0,0,0,0,0,1,0,0,0,0,0,0
3,0,114,3,61,5,0,0,0,1,0,0,0,0,0,5
4,0,66,3,17,1,0,0,0,1,0,0,0,0,0,1


## 2. Train/Test Split + Scaling

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Train attacks: {y_train.sum():,}  |  Test attacks: {y_test.sum():,}')

Train: 48,852  |  Test: 12,213
Train attacks: 20,052  |  Test attacks: 5,013


## 3. Train Random Forest

In [4]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

print('Training Random Forest (200 trees) ...')
model.fit(X_train_s, y_train)
print('Done!')

Training Random Forest (200 trees) ...


Done!


## 4. Cross-Validation

In [5]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1  = cross_val_score(model, X_train_s, y_train, cv=cv, scoring='f1')
cv_auc = cross_val_score(model, X_train_s, y_train, cv=cv, scoring='roc_auc')

print(f'5-Fold CV F1      : {cv_f1.mean():.4f}  +/-  {cv_f1.std():.4f}')
print(f'5-Fold CV ROC-AUC : {cv_auc.mean():.4f}  +/-  {cv_auc.std():.4f}')
print(f'\nPer-fold F1: {[round(v,4) for v in cv_f1]}')

5-Fold CV F1      : 0.8541  +/-  0.0017
5-Fold CV ROC-AUC : 0.9588  +/-  0.0016

Per-fold F1: [np.float64(0.8508), np.float64(0.8553), np.float64(0.8542), np.float64(0.855), np.float64(0.8553)]


## 5. Test Set Evaluation

In [6]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

THRESHOLD = 0.5
y_prob = model.predict_proba(X_test_s)[:, 1]
y_pred = (y_prob >= THRESHOLD).astype(int)

metrics = {
    'Accuracy' : accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall'   : recall_score(y_test, y_pred),
    'F1'       : f1_score(y_test, y_pred),
    'ROC-AUC'  : roc_auc_score(y_test, y_prob),
}

pd.DataFrame([metrics]).T.rename(columns={0: 'Score'}).style.background_gradient(cmap='Greens')

,Score
Accuracy,0.878163
Precision,0.814676
Recall,0.910233
F1,0.859808
ROC-AUC,0.962093


In [7]:
print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))

              precision    recall  f1-score   support

      Normal       0.93      0.86      0.89      7200
      Attack       0.81      0.91      0.86      5013

    accuracy                           0.88     12213
   macro avg       0.87      0.88      0.88     12213
weighted avg       0.88      0.88      0.88     12213



## 6. Confusion Matrix

In [8]:
from src.evaluator import plot_confusion_matrix
cm = plot_confusion_matrix(y_test, y_pred)

img = mpimg.imread('../models/confusion_matrix.png')
plt.figure(figsize=(6,5)); plt.imshow(img); plt.axis('off'); plt.show()

Saved -> C:\Users\DularaAbhiranda\Desktop\AI-WAF\ai-waf\models\confusion_matrix.png
  TP=4563  FP=1038  FN=450  TN=6162
  False Positive Rate (normal blocked): 14.42%
  False Negative Rate (attack missed) : 8.98%


## 7. ROC Curve

In [9]:
from src.evaluator import plot_roc_curve
plot_roc_curve(y_test, y_prob)

img = mpimg.imread('../models/roc_curve.png')
plt.figure(figsize=(7,5)); plt.imshow(img); plt.axis('off'); plt.show()

Saved -> C:\Users\DularaAbhiranda\Desktop\AI-WAF\ai-waf\models\roc_curve.png


## 8. Feature Importance

In [10]:
from src.evaluator import plot_feature_importance
plot_feature_importance(model, list(X.columns))

img = mpimg.imread('../models/feature_importance.png')
plt.figure(figsize=(9,6)); plt.imshow(img); plt.axis('off'); plt.show()

Saved -> C:\Users\DularaAbhiranda\Desktop\AI-WAF\ai-waf\models\feature_importance.png

Feature importances:
  url_length                0.3395
  query_length              0.1234
  body_length               0.1173
  content_length            0.1076
  path_depth                0.1011
  special_char_count        0.0965
  num_query_params          0.0286
  num_body_params           0.0262
  has_cmd_injection         0.0243
  method_is_post            0.0220
  has_sql                   0.0055
  has_xss                   0.0050
  has_null_byte             0.0029
  has_cookie                0.0000
  has_path_traversal        0.0000


## 9. Threshold Analysis

A WAF has a different cost model than a regular classifier:  
- **False Negative** (attack let through) = dangerous  
- **False Positive** (normal request blocked) = annoying but safe  

Lowering the threshold catches more attacks but also blocks more normal traffic.

In [11]:
from src.evaluator import plot_threshold_analysis
best_t = plot_threshold_analysis(y_test, y_prob)
print(f'\nRecommended threshold for this dataset: {best_t:.2f}')

img = mpimg.imread('../models/threshold_analysis.png')
plt.figure(figsize=(9,5)); plt.imshow(img); plt.axis('off'); plt.show()

Saved -> C:\Users\DularaAbhiranda\Desktop\AI-WAF\ai-waf\models\threshold_analysis.png

Best threshold by F1: 0.45  (F1=0.8602  P=0.7971  R=0.9342)

Recommended threshold for this dataset: 0.45


## 10. SHAP Explainability

SHAP (SHapley Additive exPlanations) shows *why* the model made each prediction.  
Higher SHAP value for a feature = stronger push toward predicting "Attack".

In [12]:
from src.evaluator import plot_shap
plot_shap(model, X_test_s, list(X.columns))

import os
if os.path.exists('../models/shap_summary.png'):
    img = mpimg.imread('../models/shap_summary.png')
    plt.figure(figsize=(10,7)); plt.imshow(img); plt.axis('off'); plt.show()


Calculating SHAP values (this may take ~30s) ...


Saved -> C:\Users\DularaAbhiranda\Desktop\AI-WAF\ai-waf\models\shap_summary.png


## 11. Save Model & Scaler

In [13]:
import joblib

joblib.dump(model,  '../models/model_final.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

pd.DataFrame([{
    'threshold': THRESHOLD, **metrics,
    'cv_f1_mean': cv_f1.mean(), 'cv_f1_std': cv_f1.std()
}]).to_csv('../models/eval_results.csv', index=False)

print('Saved: models/model_final.pkl')
print('Saved: models/scaler.pkl')
print('Saved: models/eval_results.csv')

Saved: models/model_final.pkl
Saved: models/scaler.pkl
Saved: models/eval_results.csv


## Summary

| Metric | Score |
|--------|-------|
| Accuracy | see above |
| Precision | see above |
| Recall | see above |
| F1 | see above |
| ROC-AUC | see above |

**Model:** Random Forest, 200 trees, `class_weight='balanced'`  
**Saved to:** `models/model_final.pkl` + `models/scaler.pkl`

**Next (Week 4):** Wire this model into a `mitmproxy` interceptor so it classifies live HTTP traffic in real time.